# Build the deduplicated locked Phase 2 artifact (Colab)

This notebook converts the existing locked **1,336-question** artifact into the human-approved **1,152-question semantic-deduplicated** artifact. It reuses the byte-identical 22,766 document chunks, BGE-M3 sparse index, and Chroma database.

> Questions do not have a persistent index in this pipeline. They are encoded at query time. This notebook rebuilds only the derived question/evidence files and validates their chunk IDs; it does **not** rebuild question embeddings or document embeddings.


In [ ]:
from pathlib import Path
import hashlib, json, os, random, shutil, subprocess, sys, time, zipfile

REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='992c0fe8877cb23aeb23314cf5d6caf52db487c5'
RAW_REPO_ID='MatchaMacchiato/newsqa_200_11064_v2.0.0'
RAW_REVISION='b81c8db6847a23272665946c0c43c72e9a212fd9'
PARENT_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-locked-v2'
PARENT_REVISION='locked-bge-m3-512-64-v2'
PARENT_FILENAME='artifacts/locked-bge-m3-512-64-v2/locked-bge-m3-512-64-v2.zip'
PARENT_ZIP_SHA256='4d79a4f016fb2fc4607c078674bbf6c80efdba3637050f7db390f21e7559cda0'
EXPECTED_CHUNKS=22766
EXPECTED_CHUNKS_SHA256='3a0f2aae08d0c978ae612c84692a3070ad2cb033935176b7c01eb7cbdc5d4498'
EXPECTED_FULL_QUESTIONS=1336
EXPECTED_DEDUP_QUESTIONS=1152
EXPECTED_DEVELOPMENT_QUESTIONS=281
EXPECTED_FINAL_QUESTIONS=871
DEVELOPMENT_ARTICLES=50
SEED=42
ARTIFACT_VERSION='locked-bge-m3-512-64-deduplicated-v2'
FORCE_REBUILD=False

CONTENT=Path('/content')
PROJECT_ROOT=CONTENT/'Text-Mining---NewsQA-RAG'
WORK_ROOT=CONTENT/'newsqa_phase2_deduplicated_build'
PARENT_ROOT=WORK_ROOT/'parent_locked'
MATERIALIZED_ROOT=WORK_ROOT/'materialized_v2'
EXPORT_ROOT=WORK_ROOT/'export'
LOG_ROOT=WORK_ROOT/'logs'
BUNDLE=CONTENT/f'{ARTIFACT_VERSION}.zip'
DRIVE_ROOT=CONTENT/'drive/MyDrive/newsqa_phase2_artifacts'
DRIVE_BUNDLE=DRIVE_ROOT/BUNDLE.name


## 1. Runtime setup

A CPU runtime is sufficient because no neural index is rebuilt. Enable Internet. Optionally add `HF_TOKEN` to Colab Secrets to avoid anonymous Hugging Face rate limits. The final ZIP is copied to Google Drive.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
try:
    HF_TOKEN=userdata.get('HF_TOKEN') or ''
except Exception:
    HF_TOKEN=''
if HF_TOKEN: os.environ['HF_TOKEN']=HF_TOKEN
else: print('No HF_TOKEN secret; both repositories are public, so anonymous download will be used.')
os.environ['HF_HOME']=str(CONTENT/'hf_cache')
os.environ.update({'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1'})

if FORCE_REBUILD and WORK_ROOT.exists(): shutil.rmtree(WORK_ROOT)
for path in [WORK_ROOT,PARENT_ROOT,MATERIALIZED_ROOT,EXPORT_ROOT,LOG_ROOT,DRIVE_ROOT]: path.mkdir(parents=True,exist_ok=True)
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=300)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
print('Pinned commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())
print('Output:',DRIVE_BUNDLE)


In [ ]:
def sha256_file(path,block_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def jsonl_rows(path):
    with Path(path).open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]
def jsonl_count(path):
    with Path(path).open(encoding='utf-8') as handle:
        return sum(1 for line in handle if line.strip())
def disk_status():
    usage=shutil.disk_usage(CONTENT); result={'free_gib':round(usage.free/2**30,2),'used_gib':round(usage.used/2**30,2),'total_gib':round(usage.total/2**30,2)}
    print('Disk:',result,flush=True); return result
def safe_extract(archive_path,target):
    target=Path(target).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            destination=(target/member.filename).resolve()
            if target not in destination.parents and destination!=target: raise RuntimeError(f'Unsafe ZIP path: {member.filename}')
        archive.extractall(target)
def run_logged(command,label):
    command=[str(v) for v in command]; log_path=LOG_ROOT/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code: raise subprocess.CalledProcessError(code,command)
    return log_path
disk_status()


## 2. Download and validate the existing locked artifact

Every checksum in its manifest is validated before any file is reused. Rerunning the notebook skips the download when the validated extraction already exists.


In [ ]:
from huggingface_hub import hf_hub_download
parent_manifest_path=PARENT_ROOT/'bundle_manifest.json'
if not parent_manifest_path.exists():
    downloaded=Path(hf_hub_download(repo_id=PARENT_REPO_ID,repo_type='dataset',revision=PARENT_REVISION,filename=PARENT_FILENAME,token=HF_TOKEN or None))
    actual_zip_sha=sha256_file(downloaded)
    assert actual_zip_sha==PARENT_ZIP_SHA256,(actual_zip_sha,PARENT_ZIP_SHA256)
    safe_extract(downloaded,PARENT_ROOT)
else:
    downloaded=None; print('Using extracted parent artifact:',PARENT_ROOT)
parent_manifest=json.loads(parent_manifest_path.read_text(encoding='utf-8'))
assert parent_manifest['artifact_version']=='locked-bge-m3-512-64-v2'
for relative,record in parent_manifest['artifacts'].items():
    path=PARENT_ROOT/relative
    assert path.is_file(),path
    assert path.stat().st_size==record['bytes'],relative
    assert sha256_file(path)==record['sha256'],relative
assert jsonl_count(PARENT_ROOT/'chunks.jsonl')==EXPECTED_CHUNKS
assert sha256_file(PARENT_ROOT/'chunks.jsonl')==EXPECTED_CHUNKS_SHA256
assert jsonl_count(PARENT_ROOT/'testset_resolved.jsonl')==EXPECTED_FULL_QUESTIONS
print('Parent artifact validated:',EXPECTED_CHUNKS,'chunks,',EXPECTED_FULL_QUESTIONS,'questions')
disk_status()


## 3. Materialize the approved v2 deduplicated testset

The canonical repository contains the semantic clusters and human approval. `--skip-vector-index` still builds the inexpensive BM25 intermediate required by the deduplication pipeline, but does not encode document vectors.


In [ ]:
dedup_testset=MATERIALIZED_ROOT/'final_deduplicated/testset_resolved.jsonl'
dedup_chunks=MATERIALIZED_ROOT/'final_deduplicated/chunks.jsonl'
if not dedup_testset.exists():
    run_logged([sys.executable,'-u','scripts/materialize_evaluation_dataset.py','--repo-id',RAW_REPO_ID,'--revision',RAW_REVISION,'--output-root',MATERIALIZED_ROOT,'--db-path',WORK_ROOT/'unused_chroma','--skip-vector-index'],'materialize_deduplicated_v2')
else:
    print('Using existing materialized deduplicated testset:',dedup_testset)
assert dedup_testset.is_file() and dedup_chunks.is_file()
print('Materialized:',jsonl_count(dedup_testset),'questions and',jsonl_count(dedup_chunks),'chunks')
disk_status()


## 4. Validate chunk identity, evidence mappings, and split

Byte-identical chunks are the condition that permits index reuse. The notebook stops here if the condition is not met.


In [ ]:
locked_chunks=PARENT_ROOT/'chunks.jsonl'
assert jsonl_count(dedup_chunks)==EXPECTED_CHUNKS
assert sha256_file(dedup_chunks)==EXPECTED_CHUNKS_SHA256,'Materialized chunks differ; do not reuse the indexes'
assert sha256_file(dedup_chunks)==sha256_file(locked_chunks)

chunk_ids={row['id'] for row in jsonl_rows(locked_chunks)}
questions=jsonl_rows(dedup_testset)
assert len(questions)==EXPECTED_DEDUP_QUESTIONS
assert len({row['question_id'] for row in questions})==len(questions)
assert len({row['article_key'] for row in questions})==200
missing={cid for row in questions for cid in row.get('relevant_chunk_ids',[]) if cid not in chunk_ids}
assert not missing,f'Missing relevant chunk IDs: {sorted(missing)[:10]}'
assert all(row.get('relevant_chunk_ids') for row in questions),'Every retained question must have mapped evidence chunks'

article_ids=sorted({row['article_key'] for row in questions})
random.Random(SEED).shuffle(article_ids)
development=set(article_ids[:DEVELOPMENT_ARTICLES])
development_count=sum(row['article_key'] in development for row in questions)
final_count=len(questions)-development_count
assert development_count==EXPECTED_DEVELOPMENT_QUESTIONS,development_count
assert final_count==EXPECTED_FINAL_QUESTIONS,final_count
print({'chunks':len(chunk_ids),'questions':len(questions),'development':development_count,'final_test':final_count,'missing_chunk_ids':len(missing)})


## 5. Assemble the deduplicated locked artifact

The neural indexes are copied unchanged. Deduplication provenance is included alongside the testset so the 1,152-question result remains auditable.


In [ ]:
if EXPORT_ROOT.exists(): shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True)
for name in ['chunks.jsonl','bge_m3_sparse.pkl']:
    shutil.copy2(PARENT_ROOT/name,EXPORT_ROOT/name)
if (PARENT_ROOT/'chroma_db').is_dir(): shutil.copytree(PARENT_ROOT/'chroma_db',EXPORT_ROOT/'chroma_db')
shutil.copy2(dedup_testset,EXPORT_ROOT/'testset_resolved.jsonl')

provenance_root=EXPORT_ROOT/'deduplication'; provenance_root.mkdir()
provenance_sources={
    'deduplicated.variant.json':MATERIALIZED_ROOT/'manifests/deduplicated.variant.json',
    'question_clusters.jsonl':MATERIALIZED_ROOT/'final_deduplicated/question_clusters.jsonl',
    'duplicate_questions.jsonl':MATERIALIZED_ROOT/'final_deduplicated/duplicate_questions.jsonl',
    'integrity_report.json':MATERIALIZED_ROOT/'final_deduplicated/integrity_report.json',
}
for name,source in provenance_sources.items(): assert source.is_file(),source; shutil.copy2(source,provenance_root/name)

files=sorted(path for path in EXPORT_ROOT.rglob('*') if path.is_file())
manifest={
    'schema_version':1,
    'artifact_version':ARTIFACT_VERSION,
    'created_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
    'source':{'hf_repo_id':RAW_REPO_ID,'hf_revision':RAW_REVISION,'repo_commit':REPO_COMMIT},
    'parent_artifact':{'hf_repo_id':PARENT_REPO_ID,'hf_revision':PARENT_REVISION,'filename':PARENT_FILENAME,'zip_sha256':PARENT_ZIP_SHA256,'artifact_version':parent_manifest['artifact_version']},
    'question_set':{'variant':'resolved','deduplication':'human_approved_within_article_semantic','primary':True},
    'pipeline':parent_manifest['pipeline'],
    'chroma':parent_manifest.get('chroma'),
    'statistics':{'chunks':EXPECTED_CHUNKS,'full_resolved_questions':EXPECTED_FULL_QUESTIONS,'resolved_questions':EXPECTED_DEDUP_QUESTIONS,'duplicate_questions_removed':EXPECTED_FULL_QUESTIONS-EXPECTED_DEDUP_QUESTIONS,'development_articles':DEVELOPMENT_ARTICLES,'development_questions':development_count,'final_test_articles':200-DEVELOPMENT_ARTICLES,'final_test_questions':final_count,'seed':SEED},
    'artifacts':{str(path.relative_to(EXPORT_ROOT)).replace('\\','/'):{'bytes':path.stat().st_size,'sha256':sha256_file(path)} for path in files},
}
(EXPORT_ROOT/'bundle_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print(json.dumps({k:v for k,v in manifest.items() if k!='artifacts'},indent=2))
print('Artifact files:',len(files)+1)


## 6. Package, verify, and save to Drive

The final cell reopens the ZIP, validates every recorded checksum, then copies it atomically to Drive. This notebook does not upload to Hugging Face.


In [ ]:
temporary=BUNDLE.with_suffix('.zip.tmp')
if temporary.exists(): temporary.unlink()
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
    for path in sorted(EXPORT_ROOT.rglob('*')):
        if path.is_file(): archive.write(path,path.relative_to(EXPORT_ROOT))
temporary.replace(BUNDLE)

with zipfile.ZipFile(BUNDLE) as archive:
    packaged=json.loads(archive.read('bundle_manifest.json'))
    assert packaged['statistics']['resolved_questions']==EXPECTED_DEDUP_QUESTIONS
    assert packaged['statistics']['development_questions']==EXPECTED_DEVELOPMENT_QUESTIONS
    assert packaged['statistics']['final_test_questions']==EXPECTED_FINAL_QUESTIONS
    for relative,record in packaged['artifacts'].items():
        info=archive.getinfo(relative); assert info.file_size==record['bytes'],relative
        assert hashlib.sha256(archive.read(relative)).hexdigest()==record['sha256'],relative

drive_temporary=DRIVE_BUNDLE.with_suffix('.zip.tmp')
shutil.copy2(BUNDLE,drive_temporary); drive_temporary.replace(DRIVE_BUNDLE)
print('Local bundle :',BUNDLE)
print('Drive bundle :',DRIVE_BUNDLE)
print('Size MiB     :',round(BUNDLE.stat().st_size/2**20,1))
print('SHA-256     :',sha256_file(BUNDLE))
print('Next step: publish under immutable tag',ARTIFACT_VERSION)
disk_status()
